# Code Converter - Python to Go

Convert Python code to optimized Go code and run it.
Reference: `week4/community-contributions/tochi/code_converter.ipynb`.

In [ ]:
# imports
import os
import io
import sys
import json
import subprocess
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display
import gradio as gr

In [ ]:
# load env + client
load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set. Check your environment variables and try again")

openai = OpenAI()
OPENAI_MODEL = "gpt-5-nano"

In [ ]:
# compile + run commands for Go
compile_command = ["go", "build", "-o", "main", "main.go"]
run_command = ["./main"]

In [ ]:
# prompts for conversion
system_prompt = """
Your task is to convert Python code into high performance Go code.
Respond only with Go code. Do not provide any explanation other than occasional comments.
The Go response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to Go with the fastest possible implementation that produces identical output in the least time.

Your response will be written to a file called main.go and then compiled and executed; the compilation command is:

{compile_command}

Respond only with Go code.
Python code to port:

```python
{python}
```
"""

In [ ]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)},
    ]

def write_output(code):
    with open("main.go", "w", encoding="utf-8") as f:
        f.write(code)

def convert(python):
    response = openai.chat.completions.create(
        model=OPENAI_MODEL,
        messages=messages_for(python),
        reasoning_effort="high",
    )
    reply = response.choices[0].message.content
    reply = reply.replace("```go", "").replace("```", "")
    return reply

In [ ]:
# sample python for testing
py_sample = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [ ]:
def run_python(code):
    globals_dict = {"__builtins__": __builtins__}
    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout

    return output


def run_go(code):
    write_output(code)
    try:
        subprocess.run(compile_command, check=True, text=True, capture_output=True)
        run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
        return run_result.stdout
    except subprocess.CalledProcessError as e:
        return f"An error occurred:\n{e.stderr}"

In [ ]:
# Gradio UI
with gr.Blocks(theme=gr.themes.Monochrome(), title="Port from Python to Go") as ui:
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python = gr.Code(
                label="Python Original Code",
                value=py_sample,
                language="python",
                lines=30,
            )
        with gr.Column(scale=6):
            go = gr.Code(
                label="Go (generated)", value="", language="go", lines=26
            )
    with gr.Row(elem_classes=["controls"]):
        python_run = gr.Button("Run Python", elem_classes=["run-btn", "py"])
        port = gr.Button("Convert to Go", elem_classes=["convert-btn"])
        go_run = gr.Button("Run Go", elem_classes=["run-btn", "go"])

    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python_out = gr.TextArea(label="Python Result", lines=10)
        with gr.Column(scale=6):
            go_out = gr.TextArea(label="Go output", lines=10)

    port.click(fn=convert, inputs=[python], outputs=[go])
    python_run.click(fn=run_python, inputs=[python], outputs=[python_out])
    go_run.click(fn=run_go, inputs=[go], outputs=[go_out])

ui.launch(inbrowser=True)